In [0]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
import utils
print ("utils imported successfully")

In [0]:
from utils.constants import *

In [0]:
def read_from_silver(spark,table):
    path = (f"{SILVER}/customers/date=2026-05-16/customers.csv")
    df = spark.read.format("delta").load(path)
    print ("loaded successfully")
    return df
gold_df_customers = read_from_silver(spark,'customers')

def read_from_silver(spark,table):
    path = (f"{SILVER}/orders/date=2026-05-16/orders.csv")
    df = spark.read.format("delta").load(path)
    print ("loaded successfully")
    return df
gold_df_orders = read_from_silver(spark,'orders')

def read_from_silver(spark,table):
    path = (f"{SILVER}/payments/date=2026-05-16/payments.csv")
    df = spark.read.format("delta").load(path)
    print ("loaded successfully")
    return df
gold_df_payments = read_from_silver(spark,'payments')




In [0]:
display(gold_df_orders)

In [0]:
def daily_sales_summary(gold_df_orders):
    window = Window\
        .orderBy(col("order_date"))\
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)

    
    
    return gold_df_orders \
        .filter(col("status")!= "cancelled")\
            .groupBy(col("order_date"))\
                .agg(
                    count("order_id").alias("total_orders"),
                    sum("total_amount").alias("total_revenue"),
                    avg("total_amount").alias("avg_order_value"),
                    max("total_amount").alias("max_order_value"),
                    min("total_amount").alias("min_order_value"),
                    countDistinct("customer_id").alias("unique_customers"),
                    countDistinct("product_id").alias("unique_products"),
                    sum(when(col("status")== "Delivered",1).otherwise(0).alias("Total_orders_delivered")),
                    sum(when(col("status")== "pending",1).otherwise(0).alias("Total_orders_pending"))
                )\
                    .withColumn(("total_revenue"),round(col("total_revenue"),2)).display()




daily_sales_summary(gold_df_orders)

   

In [0]:
gold_df_orders.filter(col("status") == "cancelled").display()

In [0]:
def customer_analytics (gold_df_customers,gold_df_orders):
    customer_orders = gold_df_orders\
        .filter(col("status")!= "cancelled")\
        .groupBy("customer_id")\
            .agg(count("order_id").alias("total_orders"),
                 sum("total_amount").alias("total_revenue"),
                 avg("total_amount").alias("avg_order_value"),
                
                 )
    df= gold_df_customers.join(gold_df_orders,on="customer_id",how ="inner")

    df = customer_orders.fillna(0,subset= ["total_orders","total_revenue","avg_order_value"])\
        .orderBy("customer_id").display()
            
    return customer_orders

customer_analytics(gold_df_customers,gold_df_orders)